# CS781 Capstone -- Phase 7: Prior Case Retrieval (PCR)

Six experiments, each starting from **one of my own Phase 1 language models** (no other pretrained model, no external API or corpus).

**Retrieval model (shared by all experiments):** a bi-encoder. Every case (query or candidate) is cut into up to `MAX_CHUNKS` chunks of `CHUNK_LEN`
tokens (head of the document), each chunk is encoded by the Phase 1 backbone (last-token + mean pooling -> linear projection -> L2 norm), and the
case embedding is the mean of its chunk embeddings. Candidates are ranked by cosine similarity and the top 7 are returned.
**Training:** contrastive InfoNCE with in-batch negatives plus one TF-IDF-mined hard negative per pair; relevant candidates of the same query are masked out of the negatives.

| # | Phase 1 model | Technique | Notes |
|---|---|---|---|
| 1 | GQA | **LoRMA** (compulsory) | own implementation |
| 2 | GPT-2 | **IA3** | learned scaling of keys, values and MLP activations, written by hand |
| 3 | SwiGLU | **Partial fine-tuning** | last 4 blocks + final norm |
| 4 | Prefix-LM | **Prefix Tuning** | learned key/value prefix through `layer_past` (run causal) |
| 5 | GQA | **Full fine-tuning** | all parameters |
| 6 | half-layer student of the best of 1-5 | **Knowledge distillation** (mandatory) | InfoNCE + KL to the teacher's similarity distribution |

Every experiment also trains the small projection head (about 0.8M parameters); `run_info.json` reports the adapter parameters and the projection separately.

**Evaluation:** micro precision / recall / F1 at 7 (primary: F1@7). Checkpoints are selected by **dev F1@7** (the competition metric).
A TF-IDF baseline, an oracle upper bound, and a zero-shot (no training) Phase 1 baseline are computed first. Each experiment writes two submissions:
`submission.zip` (dense model only) and `hybrid/submission.zip` (dense + TF-IDF score fusion, weight tuned on dev).

**Sessions:** every experiment is one cell with its own output folder; run them in separate Kaggle sessions if needed. The KD experiment needs the
teacher's `best_model.pt` (same session, or attach the earlier output and set `TEACHER_STATE_PATH` and `TEACHER_EXP`).

## 1. Hugging Face Login

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF2")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

## 2. Check GPU

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Configuration

In [ ]:
import os, random, json, time, math, zipfile, shutil, re, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PCR_FILES = {   # split -> (queries file, candidate pool file); verify with the listing cell below
    "train": ("Phase_7_PCR/Train/train.json", "Phase_7_PCR/Train/train_candidates.json"),
    "dev":   ("Phase_7_PCR/Dev/dev.json",     "Phase_7_PCR/Dev/dev_candidates.json"),
    "test":  ("Phase_7_PCR/Test/test.json",   "Phase_7_PCR/Test/test_candidates.json"),
}
MODEL_PATHS = {
    "gpt2":     "/kaggle/input/datasets/himangshumahato/model1",
    "swiglu":   "/kaggle/input/datasets/himangshumahato/model2",
    "gqa":      "/kaggle/input/datasets/himangshumahato/model3",
    "prefixlm": "/kaggle/input/datasets/himangshumahato/model4",
}
MODEL_CONTEXT_LENGTH = 1024
N_EMBD, N_LAYER, N_HEAD, N_KV_HEAD = 768, 12, 12, 4

# ---- case representation ----
CHUNK_LEN, MAX_CHUNKS, MIN_CHUNK = 384, 2, 32   # head of the document: up to MAX_CHUNKS chunks of CHUNK_LEN tokens
HEAD_CHARS = 6000                                # only this much raw text is tokenized (the BPE is slow)
TEXT_LIMIT = 30000                               # characters kept per case (TF-IDF uses all of it)
PROJ_DIM = 512
TAU = 0.05                                       # InfoNCE temperature
TOP_K = 7
MAX_PREFIX = 32                                  # largest prefix length a technique may use
assert CHUNK_LEN + MAX_PREFIX <= MODEL_CONTEXT_LENGTH

# ---- training (shared) ----
PAIRS_PER_BATCH = 8                              # (query, positive) pairs per step; each pair also gets 1 hard negative
PAIRS_PER_EPOCH = 6000                           # random pairs per epoch (lower it if Kaggle time runs short)
HARD_NEG, HARD_NEG_K = True, 30                  # hard negatives come from the TF-IDF top-K non-relevant candidates
DEV_POOL_CAP = 1000                              # per-epoch dev eval uses all gold candidates + random others up to this size
WEIGHT_DECAY, WARMUP_RATIO = 0.01, 0.10
HYBRID_ALPHAS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]   # 1.0 = dense only, 0.0 = TF-IDF only

# switch experiments on/off (each can run in its own session)
RUN = {"e1": True, "e2": True, "e3": True, "e4": True, "e5": True, "e6": True}
RUN_BASELINES = True
TEACHER_STATE_PATH = None                        # saved best_model.pt of the teacher if it ran in another session
TEACHER_EXP = None                               # "e1".."e5"; None = best finished experiment by dev F1@7

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BASE_OUTPUT_DIR = "/kaggle/working/cs781_phase7"
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

def experiment_dirs(name):
    d = os.path.join(BASE_OUTPUT_DIR, name)
    os.makedirs(d, exist_ok=True)
    return {"dir": d, "model_path": os.path.join(d, "best_model.pt"),
            "run_info_path": os.path.join(d, "run_info.json"),
            "dense_dir": d, "hybrid_dir": os.path.join(d, "hybrid")}

print("Configuration loaded.")

## 4. Install helpers

In [ ]:
!pip -q install regex

## 5. Tokenizer (same shared tokenizer as Phases 1-6)

In [ ]:
%%writefile a1_tokenizer.py
"""From-scratch byte-level BPE tokenizer (GPT-2 style), no external tokenizer libs.

Differences from the plain-bytes version:
  * Text is first split into "words" with a GPT-2 style regex, and merges never
    cross word boundaries. Training and encoding then work on *unique words*
    (with counts) instead of the whole corpus, which is orders of magnitude faster.
  * Training uses incremental pair counts + a lazy max-heap (no full rescan per merge).
  * Encoding applies merges per word by rank, with a word -> ids cache.
  * One special token, <|endoftext|>, used as EOS / PAD / BOS.

Interface:
    tok.encode(text, add_eos=False) -> list[int]
    tok.decode(ids, skip_special_tokens=True) -> str
    tok.vocab_size / tok.get_vocab_size()
    tok.eos_token_id, tok.pad_token_id, tok.bos_token_id
    tok.save(path) / ByteLevelBPE.load(path)
"""

from __future__ import annotations

import heapq
import json
import time
from collections import Counter, defaultdict

import regex as re  # `pip install regex` (needed for \p{L} / \p{N})

# GPT-2 pre-tokenization pattern
_PAT = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

EOS_TOKEN = "<|endoftext|>"


class ByteLevelBPE:
    def __init__(self):
        # ids 0-255 = raw bytes; 256.. = learned merges; last id = <|endoftext|>
        self.merges: dict[tuple[int, int], int] = {}      # (a, b) -> new_id (id order == rank order)
        self.itos: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
        self.eos_token_id = 256
        self.vocab_size = 257
        self._cache: dict[str, tuple] = {}

    # alias attributes so the tokenizer plugs into HF-style code
    @property
    def pad_token_id(self): return self.eos_token_id
    @property
    def bos_token_id(self): return self.eos_token_id

    def __len__(self): return self.vocab_size
    def get_vocab_size(self): return self.vocab_size

    def train(self, text: str, vocab_size: int, verbose: bool = False) -> None:
        """Learn (vocab_size - 257) merges; the final id is reserved for <|endoftext|>."""
        if vocab_size < 257:
            raise ValueError("vocab_size must be at least 257 (256 bytes + EOS)")
        num_merges = vocab_size - 257
        t0 = time.time()

        # 1) pre-tokenize and count unique words
        word_counts = Counter(_PAT.findall(text))
        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        if verbose:
            print(f"[bpe] {len(words):,} unique words from {sum(freqs):,} total words")

        # 2) initial pair statistics
        stats: dict[tuple[int, int], int] = defaultdict(int)
        where: dict[tuple[int, int], set] = defaultdict(set)
        for wi, w in enumerate(words):
            f = freqs[wi]
            for p in zip(w, w[1:]):
                stats[p] += f
                where[p].add(wi)

        # lazy max-heap: (-count, a, b) -> highest count first, ties -> smallest (a, b)
        heap = [(-c, a, b) for (a, b), c in stats.items()]
        heapq.heapify(heap)

        self.merges, self.itos = {}, {i: bytes([i]) for i in range(256)}

        # 3) merge loop
        for i in range(num_merges):
            pair, count = None, 0
            while heap:
                negc, a, b = heapq.heappop(heap)
                if stats.get((a, b), 0) == -negc:   # skip stale heap entries
                    pair, count = (a, b), -negc
                    break
            if pair is None or count < 2:
                break

            new_id = 256 + i
            self.merges[pair] = new_id
            self.itos[new_id] = self.itos[pair[0]] + self.itos[pair[1]]

            touched = set()
            for wi in list(where[pair]):
                w, f = words[wi], freqs[wi]
                # remove this word's old pair contributions
                for p in zip(w, w[1:]):
                    stats[p] -= f
                    where[p].discard(wi)
                    touched.add(p)
                # apply merge (leftmost, non-overlapping)
                a, b = pair
                out, j, n = [], 0, len(w)
                while j < n:
                    if j < n - 1 and w[j] == a and w[j + 1] == b:
                        out.append(new_id); j += 2
                    else:
                        out.append(w[j]); j += 1
                words[wi] = out
                # add new pair contributions
                for p in zip(out, out[1:]):
                    stats[p] += f
                    where[p].add(wi)
                    touched.add(p)

            for p in touched:
                c = stats.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p[0], p[1]))
                else:
                    stats.pop(p, None)
                    where.pop(p, None)

            if verbose and (i % 1000 == 0 or i == num_merges - 1):
                print(f"[bpe] merge {i + 1}/{num_merges} | pair={pair} count={count:,} "
                      f"| elapsed={time.time() - t0:.1f}s")

        self.eos_token_id = 256 + len(self.merges)
        self.vocab_size = self.eos_token_id + 1
        self._cache = {}

    # Inference
    def _encode_word(self, word: str) -> tuple:
        hit = self._cache.get(word)
        if hit is not None:
            return hit
        ids = list(word.encode("utf-8"))
        merges = self.merges
        while len(ids) > 1:
            # pick the applicable merge learned earliest (lowest id)
            best, best_id = None, None
            for p in zip(ids, ids[1:]):
                nid = merges.get(p)
                if nid is not None and (best_id is None or nid < best_id):
                    best, best_id = p, nid
            if best is None:
                break
            a, b = best
            out, j, n = [], 0, len(ids)
            while j < n:
                if j < n - 1 and ids[j] == a and ids[j + 1] == b:
                    out.append(best_id); j += 2
                else:
                    out.append(ids[j]); j += 1
            ids = out
        res = tuple(ids)
        if len(self._cache) > 2_000_000:
            self._cache.clear()
        self._cache[word] = res
        return res

    def encode(self, text: str, add_eos: bool = False) -> list[int]:
        ids: list[int] = []
        for w in _PAT.findall(text):
            ids.extend(self._encode_word(w))
        if add_eos:
            ids.append(self.eos_token_id)
        return ids

    def decode(self, ids, skip_special_tokens: bool = True) -> str:
        if hasattr(ids, "tolist"):
            ids = ids.tolist()
        parts = []
        for i in ids:
            if i == self.eos_token_id:
                if not skip_special_tokens:
                    parts.append(EOS_TOKEN.encode("utf-8"))
                continue
            parts.append(self.itos[i])
        return b"".join(parts).decode("utf-8", errors="replace")

    def save(self, path: str) -> None:
        data = {
            "merges": [[a, b, nid] for (a, b), nid in self.merges.items()],
            "vocab_size": self.vocab_size,
            "eos_token_id": self.eos_token_id,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    @classmethod
    def load(cls, path: str) -> "ByteLevelBPE":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        for a, b, nid in data["merges"]:
            tok.merges[(a, b)] = nid
            tok.itos[nid] = tok.itos[a] + tok.itos[b]
        tok.eos_token_id = data["eos_token_id"]
        tok.vocab_size = data["vocab_size"]
        return tok


def train_tokenizer(train_text: str, vocab_size: int):
    """Train a byte-level BPE tokenizer on `train_text` and return it."""
    tok = ByteLevelBPE()
    tok.train(train_text, vocab_size, verbose=True)
    return tok

In [ ]:
from a1_tokenizer import ByteLevelBPE, _PAT

tokenizer = ByteLevelBPE.load(os.path.join(MODEL_PATHS["gqa"], "bpe_tokenizer.json"))
print("Vocabulary size:", len(tokenizer))
for name, path in MODEL_PATHS.items():
    t = ByteLevelBPE.load(os.path.join(path, "bpe_tokenizer.json"))
    assert t.vocab_size == tokenizer.vocab_size, f"{name} tokenizer differs!"
print("All four checkpoints share the same tokenizer.")
EOS_ID = tokenizer.eos_token_id
PAD_ID = EOS_ID

## 6. Load the Phase 7 data (verify the file paths first)

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
info = HfApi().dataset_info(HF_DATASET_REPO, files_metadata=True, token=HF_TOKEN)
for s in info.siblings:
    if "Phase_7" in s.rfilename:
        print(f"{s.rfilename:60s} {s.size/1e6:.2f} MB")

In [ ]:
ID_KEYS   = ["id", "case_id", "doc_id", "document_id"]
TEXT_KEYS = ["text", "document", "case_text", "sentences"]
REL_KEYS  = ["relevant_candidates", "relevant", "cited_cases", "labels"]

def _get(d, keys):
    for k in keys:
        if k in d:
            return d[k]
    return None

def doc_text(x, limit):
    """Case text (a string or a list of sentences) -> one string of at most `limit` characters."""
    if isinstance(x, str):
        return x[:limit]
    if isinstance(x, (list, tuple)):
        out, n = [], 0
        for s in x:
            s = s if isinstance(s, str) else str(s)
            out.append(s); n += len(s) + 1
            if n >= limit:
                break
        return " ".join(out)[:limit]
    return str(x)[:limit]

def _items(raw):
    return list(raw.items()) if isinstance(raw, dict) else [(None, v) for v in raw]

def to_pool(raw, limit):
    """Candidate pool -> (ids, texts). Accepts a list of dicts or a dict id -> dict / text."""
    ids, texts = [], []
    for k, v in _items(raw):
        if isinstance(v, dict):
            rid = _get(v, ID_KEYS); rid = k if rid is None else rid
            txt = _get(v, TEXT_KEYS)
            if txt is None:
                raise KeyError(f"no text field in a candidate; keys: {list(v.keys())}")
        else:
            rid, txt = k, v
        if rid is None:
            raise KeyError("candidate without an id")
        ids.append(str(rid)); texts.append(doc_text(txt, limit))
    return ids, texts

def to_queries(raw, limit, with_labels):
    out = []
    for k, v in _items(raw):
        if not isinstance(v, dict):
            raise ValueError("a query record must be a dict")
        qid = _get(v, ID_KEYS); qid = k if qid is None else qid
        txt = _get(v, TEXT_KEYS)
        if txt is None:
            raise KeyError(f"no text field in a query; keys: {list(v.keys())}")
        rel = _get(v, REL_KEYS) if with_labels else []
        out.append({"id": str(qid), "text": doc_text(txt, limit), "rel": [str(r) for r in (rel or [])]})
    return out

In [ ]:
from tqdm.auto import tqdm

def hf_json(path):
    p = hf_hub_download(repo_id=HF_DATASET_REPO, filename=path, repo_type="dataset", token=HF_TOKEN)
    with open(p, encoding="utf-8") as f:
        return json.load(f)

def chunk_doc(text):
    """Head of the case -> up to MAX_CHUNKS chunks of CHUNK_LEN token ids (a short trailing chunk is dropped)."""
    ids = tokenizer.encode(text[:HEAD_CHARS])[:CHUNK_LEN * MAX_CHUNKS]
    chunks = [ids[i:i + CHUNK_LEN] for i in range(0, len(ids), CHUNK_LEN)]
    if len(chunks) > 1 and len(chunks[-1]) < MIN_CHUNK:
        chunks.pop()
    return chunks or [[EOS_ID]]


def tfidf_score_matrix(q_texts, pool_texts):
    """Lexical baseline / hard-negative miner: cosine similarity of TF-IDF vectors, [n_queries, n_pool]."""
    from sklearn.feature_extraction.text import TfidfVectorizer
    vec = TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.5, max_features=300000, dtype=np.float32)
    vec.fit(pool_texts + q_texts)
    Xq, Xc = vec.transform(q_texts), vec.transform(pool_texts)
    return (Xq @ Xc.T).toarray().astype(np.float32)


class Split:
    """Queries + their own candidate pool (a split's queries must only retrieve from its pool)."""
    def __init__(self, name, q_raw, pool_raw, with_labels):
        self.name = name
        self.pool_ids, self.pool_texts = to_pool(pool_raw, TEXT_LIMIT)
        self.pool_index = {d: i for i, d in enumerate(self.pool_ids)}
        self.queries = to_queries(q_raw, TEXT_LIMIT, with_labels)
        missing = 0
        for q in self.queries:
            q["rel_idx"] = [self.pool_index[r] for r in q["rel"] if r in self.pool_index]
            missing += len(q["rel"]) - len(q["rel_idx"])
            q["self_idx"] = self.pool_index.get(q["id"], -1)      # the query itself must never be returned
        self.n_missing_gold = missing
        self._q_chunks = self._pool_chunks = self._tfidf = None

    @property
    def q_chunks(self):
        if self._q_chunks is None:
            self._q_chunks = [chunk_doc(q["text"]) for q in tqdm(self.queries, desc=f"{self.name} queries")]
        return self._q_chunks

    @property
    def pool_chunks(self):
        if self._pool_chunks is None:
            self._pool_chunks = [chunk_doc(t) for t in tqdm(self.pool_texts, desc=f"{self.name} pool")]
        return self._pool_chunks

    def tfidf_scores(self):
        if self._tfidf is None:
            self._tfidf = tfidf_score_matrix([q["text"] for q in self.queries], self.pool_texts)
        return self._tfidf

    def gold(self):
        return {q["id"]: set(q["rel"]) for q in self.queries}


splits = {}
for name, (qf, cf) in PCR_FILES.items():
    splits[name] = Split(name, hf_json(qf), hf_json(cf), with_labels=(name != "test"))
    s = splits[name]
    n_gold = [len(q["rel"]) for q in s.queries]
    print(f"{name:5s}: {len(s.queries):4d} queries | pool {len(s.pool_ids):5d} | "
          + (f"gold per query mean {np.mean(n_gold):.1f} (min {min(n_gold)}, max {max(n_gold)}) | gold missing from pool: {s.n_missing_gold}" if name != "test" else "no labels"))
    assert len(set(s.pool_ids)) == len(s.pool_ids), "duplicate candidate ids"
train_split, dev_split, test_split = splits["train"], splits["dev"], splits["test"]
print("query ids that also appear in their own pool (train/dev/test):",
      [sum(q["self_idx"] >= 0 for q in splits[n].queries) for n in ("train", "dev", "test")])

## 7. Metrics, ranking and score fusion

Micro-F1@7 pools all queries: precision = hits / (7 x queries), recall = hits / (all gold candidates, not truncated), F1 = their harmonic mean.
The oracle row is the best possible F1@7 given the gold-set sizes (a query with more than 7 gold candidates cannot reach full recall).

In [ ]:
def f1_at_k(preds, golds, k=TOP_K):
    """preds: {qid: [ids]}; golds: {qid: set(ids)} -> dict(p, r, f1), micro-averaged over queries."""
    tp = n_pred = n_gold = 0
    for q, g in golds.items():
        p = list(preds.get(q, []))[:k]
        tp += len(set(p) & g); n_pred += len(p); n_gold += len(g)
    P = tp / n_pred if n_pred else 0.0
    R = tp / n_gold if n_gold else 0.0
    F = 2 * P * R / (P + R) if P + R else 0.0
    return {"p": P, "r": R, "f1": F}

def oracle_f1_at_k(golds, k=TOP_K):
    tp = sum(min(k, len(g)) for g in golds.values())
    n_pred, n_gold = k * len(golds), sum(len(g) for g in golds.values())
    P, R = tp / n_pred, tp / n_gold
    return {"p": P, "r": R, "f1": 2 * P * R / (P + R)}

def zrow(m):
    return (m - m.mean(1, keepdims=True)) / (m.std(1, keepdims=True) + 1e-6)

def hybrid_scores(dense, tfidf, alpha):
    """alpha * z(dense) + (1 - alpha) * z(tfidf); alpha = 1 keeps the dense ranking."""
    return alpha * zrow(dense) + (1 - alpha) * zrow(tfidf)

def topk_ids(score_row, ids, exclude_pos=-1, k=TOP_K):
    s = np.array(score_row, dtype=np.float64)
    if exclude_pos >= 0:
        s[exclude_pos] = -1e18
    if len(s) <= k:
        idx = np.argsort(-s)
    else:
        idx = np.argpartition(-s, k)[:k]
        idx = idx[np.argsort(-s[idx])]
    return [ids[i] for i in idx[:k]]

def predictions_from_scores(split, scores, pool_idx=None):
    """scores: [n_queries, len(pool_idx or full pool)] -> {query id: top-7 candidate ids}."""
    pool_idx = list(range(len(split.pool_ids))) if pool_idx is None else pool_idx
    ids = [split.pool_ids[i] for i in pool_idx]
    pos = {p: i for i, p in enumerate(pool_idx)}
    return {q["id"]: topk_ids(scores[qi], ids, pos.get(q["self_idx"], -1)) for qi, q in enumerate(split.queries)}

def eval_scores(split, scores, pool_idx=None):
    return f1_at_k(predictions_from_scores(split, scores, pool_idx), split.gold())

def make_small_pool(split, cap, seed=0):
    """All gold candidates of the split's queries plus random others, up to `cap` documents."""
    gold = {i for q in split.queries for i in q["rel_idx"]}
    others = [i for i in range(len(split.pool_ids)) if i not in gold]
    np.random.RandomState(seed).shuffle(others)
    return sorted(list(gold) + others[:max(0, cap - len(gold))])

def tune_hybrid(split, dense, tfidf, pool_idx=None):
    tf = tfidf if pool_idx is None else tfidf[:, pool_idx]
    best = None
    for a in HYBRID_ALPHAS:
        r = eval_scores(split, hybrid_scores(dense, tf, a), pool_idx)
        if best is None or r["f1"] >= best[1]["f1"] - 1e-12 and (r["f1"] > best[1]["f1"] or a > best[0]):
            best = (a, r)
    return best

def fmt(r):
    return f"P@7 {r['p']:.4f} | R@7 {r['r']:.4f} | F1@7 {r['f1']:.4f}"

def validate_predictions(preds, split):
    pool = set(split.pool_ids)
    assert set(preds) == {q["id"] for q in split.queries}, "predictions must cover every query"
    for qid, ids in preds.items():
        assert len(ids) == TOP_K and len(set(ids)) == TOP_K, f"{qid}: need {TOP_K} unique ids"
        assert all(i in pool for i in ids), f"{qid}: id outside the candidate pool"

def write_submission(out_dir, preds, split):
    validate_predictions(preds, split)
    os.makedirs(out_dir, exist_ok=True)
    pj, zp = os.path.join(out_dir, "prediction.json"), os.path.join(out_dir, "submission.zip")
    with open(pj, "w") as f:
        json.dump(preds, f)
    with zipfile.ZipFile(zp, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    with zipfile.ZipFile(zp) as z:
        assert z.namelist() == ["prediction.json"]
    print("saved:", zp, "| queries:", len(preds))

## 8. Reference points: oracle, TF-IDF, and zero-shot Phase 1 embeddings (run after section 10)

These give the report its baselines: the best possible F1@7, a lexical retriever, and how well the *untrained* Phase 1 representations (mean-pooled hidden states) retrieve legal cases.

## 9. Backbone architectures (same four classes as Phases 2-6) and loader

In [ ]:
import math as _m

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.n_head = n_head; self.n_embd = n_embd
        self.head_dim = n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout); self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(n_positions, n_positions)).view(1, 1, n_positions, n_positions))

    def forward(self, x, layer_past=None, attn_mask=None):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        if layer_past is not None:
            pk, pv = layer_past; k = torch.cat([pk, k], -2); v = torch.cat([pv, v], -2)
        present = (k, v)
        past_len = layer_past[0].size(-2) if layer_past is not None else 0
        seq_len = T + past_len
        att = (q @ k.transpose(-2, -1)) * (1.0 / _m.sqrt(self.head_dim))
        if attn_mask is not None:
            att = att.masked_fill(~attn_mask, float('-inf'))
        else:
            mask = self.bias[:, :, past_len:past_len + T, :seq_len] == 0
            att = att.masked_fill(mask, float('-inf'))
        att = self.attn_dropout(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd); self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * n_embd, n_embd); self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd); self.attn = CausalSelfAttention(n_embd, n_head, n_positions, dropout)
        self.ln_2 = nn.LayerNorm(n_embd); self.mlp = MLP(n_embd, dropout)

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class GPT2ScratchLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([TransformerBlock(n_embd, n_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


class SwiGLUMLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1, hidden_mult=4):
        super().__init__()
        hidden = int(n_embd * hidden_mult * 2 / 3)
        self.w1 = nn.Linear(n_embd, hidden, bias=False)
        self.w3 = nn.Linear(n_embd, hidden, bias=False)
        self.w2 = nn.Linear(hidden, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))


class CausalSelfAttentionRMS(CausalSelfAttention):
    pass


class RMSSwiGLUBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = RMSNorm(n_embd); self.attn = CausalSelfAttentionRMS(n_embd, n_head, n_positions, dropout)
        self.ln_2 = RMSNorm(n_embd); self.mlp = SwiGLUMLP(n_embd, dropout)

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class RMSSwiGLULM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([RMSSwiGLUBlock(n_embd, n_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, n_positions, dropout=0.1):
        super().__init__()
        self.n_head = n_head; self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head; self.num_groups = n_head // n_kv_head
        self.q_proj = nn.Linear(n_embd, n_embd)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout); self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(n_positions, n_positions)).view(1, 1, n_positions, n_positions))

    def forward(self, x, layer_past=None, attn_mask=None):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        if layer_past is not None:
            pk, pv = layer_past; k = torch.cat([pk, k], -2); v = torch.cat([pv, v], -2)
        present = (k, v)
        past_len = layer_past[0].size(-2) if layer_past is not None else 0
        seq_len = T + past_len
        k = k.repeat_interleave(self.num_groups, dim=1); v = v.repeat_interleave(self.num_groups, dim=1)
        att = (q @ k.transpose(-2, -1)) * (1.0 / _m.sqrt(self.head_dim))
        if attn_mask is not None:
            att = att.masked_fill(~attn_mask, float('-inf'))
        else:
            mask = self.bias[:, :, past_len:past_len + T, :seq_len] == 0
            att = att.masked_fill(mask, float('-inf'))
        att = self.attn_dropout(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class GQATransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, n_positions, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.GELU(approximate='tanh'),
                                 nn.Linear(4*n_embd, n_embd), nn.Dropout(dropout))

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class ScratchGroupedQueryLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, n_kv_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([GQATransformerBlock(n_embd, n_head, n_kv_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


class PrefixLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, n_kv_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([GQATransformerBlock(n_embd, n_head, n_kv_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

print("Backbone classes defined.")

In [ ]:
from safetensors.torch import load_file

BACKBONE_REGISTRY = {
    "gpt2":     {"cls": GPT2ScratchLM,         "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD)},
    "swiglu":   {"cls": RMSSwiGLULM,           "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD)},
    "gqa":      {"cls": ScratchGroupedQueryLM, "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD, n_kv_head=N_KV_HEAD)},
    "prefixlm": {"cls": PrefixLM,              "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD, n_kv_head=N_KV_HEAD)},
}

def load_lm(name):
    spec = BACKBONE_REGISTRY[name]
    model = spec["cls"](vocab_size=tokenizer.vocab_size, n_positions=MODEL_CONTEXT_LENGTH, **spec["kwargs"])
    state = load_file(os.path.join(MODEL_PATHS[name], "model.safetensors"))
    print(f"[{name}] load_state_dict:", model.load_state_dict(state))
    return model

def build_student_lm(backbone):
    """Half the layers (0,2,4,...) of a Phase 1 backbone, initialised from those Phase 1 layers."""
    reg = BACKBONE_REGISTRY[backbone]
    keep = list(range(0, N_LAYER, 2))
    kwargs = dict(reg["kwargs"]); kwargs["n_layer"] = len(keep)
    lm = reg["cls"](vocab_size=tokenizer.vocab_size, n_positions=MODEL_CONTEXT_LENGTH, **kwargs)
    state = load_file(os.path.join(MODEL_PATHS[backbone], "model.safetensors"))
    new = {}
    for k, v in state.items():
        if k.startswith("h."):
            parts = k.split(".")
            if int(parts[1]) in keep:
                parts[1] = str(keep.index(int(parts[1])))
                new[".".join(parts)] = v
        else:
            new[k] = v
    print("[student] load_state_dict:", lm.load_state_dict(new))
    return lm

for _name in BACKBONE_REGISTRY:
    _tmp = load_lm(_name)
    print(f"{_name}: {sum(p.numel() for p in _tmp.parameters())/1e6:.1f}M params")
    del _tmp

## 10. Case encoder

`DocEncoder` = Phase 1 backbone (language-model head removed) + pooling + projection. Chunks are right-padded; the backbone is causal, so padding never affects real tokens.
Pooling concatenates the last real token's state and the mean over real tokens. `raw=True` (zero-shot baseline) uses the mean-pooled hidden state without a projection.
Prefix Tuning feeds learned per-layer key/values to the attention layers through `layer_past`; they consume no position embeddings.

In [ ]:
class DocEncoder(nn.Module):
    def __init__(self, lm, n_embd=N_EMBD, proj_dim=PROJ_DIM, n_prefix=0, raw=False):
        super().__init__()
        if hasattr(lm, "lm_head"):
            del lm.lm_head                      # retrieval does not predict tokens
        self.lm, self.raw, self.n_prefix = lm, raw, n_prefix
        self.proj = nn.Linear(2 * n_embd, proj_dim)
        self.prefix_k = self.prefix_v = None
        if n_prefix > 0:
            attn = lm.h[0].attn
            n_kv = getattr(attn, "n_kv_head", attn.n_head)
            shape = (len(lm.h), n_kv, n_prefix, attn.head_dim)
            self.prefix_k = nn.Parameter(torch.randn(shape) * 0.02)
            self.prefix_v = nn.Parameter(torch.randn(shape) * 0.02)

    def _prefix_past(self, B):
        dt = torch.float16 if (torch.is_autocast_enabled() and torch.cuda.is_available()) else torch.float32
        return [(self.prefix_k[i].unsqueeze(0).expand(B, -1, -1, -1).to(dt),
                 self.prefix_v[i].unsqueeze(0).expand(B, -1, -1, -1).to(dt)) for i in range(self.prefix_k.size(0))]

    def hidden(self, ids):
        B, T = ids.shape
        past = self._prefix_past(B) if self.prefix_k is not None else None
        x = self.lm.wte(ids) + self.lm.wpe(torch.arange(T, device=ids.device))
        x = self.lm.drop(x)
        for i, block in enumerate(self.lm.h):
            x, _ = block(x, layer_past=None if past is None else past[i])
        return self.lm.ln_f(x)

    def forward_chunks(self, ids, lengths):
        """ids [N, L] (right-padded), lengths [N] -> L2-normalised chunk embeddings [N, D]."""
        h = self.hidden(ids)
        N, L = ids.shape
        mask = (torch.arange(L, device=ids.device)[None, :] < lengths[:, None]).float().unsqueeze(-1)
        mean = (h.float() * mask).sum(1) / mask.sum(1).clamp(min=1)
        if self.raw:
            emb = mean
        else:
            last = h[torch.arange(N, device=ids.device), lengths - 1].float()
            emb = self.proj(torch.cat([last, mean], dim=-1))
        return F.normalize(emb, dim=-1)

print("DocEncoder defined.")

In [ ]:
def pad_chunks(chunks):
    L = max(len(c) for c in chunks)
    ids = torch.full((len(chunks), L), PAD_ID, dtype=torch.long)
    for i, c in enumerate(chunks):
        ids[i, :len(c)] = torch.tensor(c)
    return ids, torch.tensor([len(c) for c in chunks])

def encode_docs_train(model, docs_chunks):
    """Case embeddings WITH gradients: mean of the chunk embeddings, re-normalised."""
    flat = [c for ch in docs_chunks for c in ch]
    counts = [len(ch) for ch in docs_chunks]
    ids, lengths = pad_chunks(flat)
    emb = model.forward_chunks(ids.to(device), lengths.to(device))
    parts = torch.split(emb, counts)
    return F.normalize(torch.stack([p.mean(0) for p in parts]), dim=-1)

@torch.no_grad()
def embed_chunks_all(model, docs_chunks, bs=32):
    """Chunk embeddings of many cases, no gradients. Returns (emb [n_chunks, D] float32, counts per case)."""
    model.eval()
    flat = [c for ch in docs_chunks for c in ch]
    order = sorted(range(len(flat)), key=lambda k: len(flat[k]))
    out = [None] * len(flat)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        ids, lengths = pad_chunks([flat[k] for k in idx])
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            e = model.forward_chunks(ids.to(device), lengths.to(device))
        for j, k in enumerate(idx):
            out[k] = e[j].float()
    return torch.stack(out), [len(ch) for ch in docs_chunks]

def aggregate(emb, counts, mode="mean"):
    parts = torch.split(emb, counts)
    docs = torch.stack([p.mean(0) for p in parts]) if mode == "mean" else torch.stack([p[0] for p in parts])
    return F.normalize(docs, dim=-1)

def maxsim_scores(q_emb, q_counts, c_emb, c_counts):
    """Late interaction: for each query chunk the best-matching candidate chunk, averaged over the query's chunks."""
    S = q_emb @ c_emb.T
    cols = torch.stack([b.max(dim=1).values for b in torch.split(S, c_counts, dim=1)], dim=1)
    return torch.stack([b.mean(0) for b in torch.split(cols, q_counts, dim=0)])

@torch.no_grad()
def dense_scores(model, split, pool_idx=None, mode="mean"):
    """Cosine scores [n_queries, len(pool)] (numpy). Returns (scores, pool_idx used)."""
    pool_idx = list(range(len(split.pool_ids))) if pool_idx is None else pool_idx
    q_emb, qc = embed_chunks_all(model, split.q_chunks)
    c_emb, cc = embed_chunks_all(model, [split.pool_chunks[i] for i in pool_idx])
    if mode == "maxsim":
        S = maxsim_scores(q_emb, qc, c_emb, cc)
    else:
        S = aggregate(q_emb, qc, mode) @ aggregate(c_emb, cc, mode).T
    return S.cpu().numpy(), pool_idx

In [ ]:
if RUN_BASELINES:
    gold_dev = dev_split.gold()
    print("oracle  :", fmt(oracle_f1_at_k(gold_dev)))
    S_tf = dev_split.tfidf_scores()
    print("TF-IDF  :", fmt(eval_scores(dev_split, S_tf)))
    for _name in BACKBONE_REGISTRY:                      # zero-shot Phase 1 representations (no training at all)
        _enc = DocEncoder(load_lm(_name), raw=True).to(device)
        S0, _ = dense_scores(_enc, dev_split)
        print(f"zero-shot {_name:8s}:", fmt(eval_scores(dev_split, S0)))
        del _enc; gc.collect(); torch.cuda.empty_cache()

## 11. Adaptation techniques (all hand-written)

LoRMA (multiplicative low-rank update, same layer as Phases 2-6); IA3 (learned scaling vectors on keys, values and the MLP's intermediate activations);
partial fine-tuning (last blocks + final norm); Prefix Tuning lives inside `DocEncoder`; full fine-tuning unfreezes everything.

In [ ]:
class LoRMALinear(nn.Module):
    """LoRMA: W' = (I + (alpha/r) B A) W0 (multiplicative low-rank update)."""
    def __init__(self, base_linear, r=8, alpha=16):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad = False
        d_out = base_linear.out_features
        self.r = r; self.alpha = alpha
        self.B = nn.Parameter(torch.zeros(d_out, r))
        self.A = nn.Parameter(torch.randn(r, d_out) * 0.01)
        self.register_buffer("eye", torch.eye(d_out))

    def forward(self, x):
        M = self.eye + (self.alpha / self.r) * (self.B @ self.A)
        return F.linear(x, M @ self.base.weight, self.base.bias)

def apply_lorma(model, target_names, r=8, alpha=16):
    for parent_name, parent in list(model.named_modules()):
        for child_name, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and child_name in target_names:
                setattr(parent, child_name, LoRMALinear(child, r=r, alpha=alpha))
    return model


class IA3QKV(nn.Module):
    """IA3 on the fused q/k/v projection: keys and values are rescaled by learned vectors (initialised to 1)."""
    def __init__(self, base, n_embd):
        super().__init__()
        self.base, self.n_embd = base, n_embd
        for p in self.base.parameters():
            p.requires_grad = False
        self.l_k = nn.Parameter(torch.ones(n_embd))
        self.l_v = nn.Parameter(torch.ones(n_embd))

    def forward(self, x):
        q, k, v = self.base(x).split(self.n_embd, dim=-1)
        return torch.cat([q, k * self.l_k, v * self.l_v], dim=-1)

class IA3Input(nn.Module):
    """IA3 on the MLP: the intermediate activation is rescaled before the output projection."""
    def __init__(self, base, dim):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.l_ff = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return self.base(x * self.l_ff)

def apply_ia3(lm, n_embd):
    """For the GPT-2 style blocks (fused c_attn, MLP with c_proj)."""
    for blk in lm.h:
        blk.attn.c_attn = IA3QKV(blk.attn.c_attn, n_embd)
        blk.mlp.c_proj = IA3Input(blk.mlp.c_proj, 4 * n_embd)
    return lm

print("LoRMA and IA3 defined. (Prefix Tuning is inside DocEncoder.)")

## 12. Contrastive training (with optional distillation) and the experiment runner

Each step takes `PAIRS_PER_BATCH` (query, relevant candidate) pairs and adds one hard negative per pair (TF-IDF top-`HARD_NEG_K` non-relevant candidates).
Every query is scored against all positives and hard negatives in the batch; candidates that are relevant to that query (or the query itself) are masked out of its negatives.
The epoch is selected by **dev F1@7** on a reduced dev pool (all gold candidates + random others); the final report uses the full dev pool.
For distillation the loss is `alpha * InfoNCE + (1 - alpha) * T^2 * KL(teacher similarity distribution || student)` over the same candidate set.

In [ ]:
def trainable_state(model):
    return {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}

def load_trainable(model, path):
    state = torch.load(path, map_location="cpu")
    res = model.load_state_dict(state, strict=False)
    assert not res.unexpected_keys, res.unexpected_keys
    return model

def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tr, sum(p.numel() for p in model.parameters())

def build_hard_negatives(split, k=HARD_NEG_K):
    S = split.tfidf_scores()
    hn = []
    for qi, q in enumerate(split.queries):
        rel = set(q["rel_idx"]) | {q["self_idx"]}
        order = np.argsort(-S[qi])
        hn.append([int(i) for i in order if int(i) not in rel][:k])
    return hn

def train_retriever(model, groups, spec, paths, teacher=None):
    tr = train_split
    pairs = [(qi, p) for qi, q in enumerate(tr.queries) for p in q["rel_idx"]]
    assert pairs, "no (query, relevant candidate) pairs in the training split"
    hardneg = build_hard_negatives(tr) if HARD_NEG else None
    epochs, patience = spec["epochs"], spec["patience"]
    steps_per_epoch = math.ceil(min(len(pairs), PAIRS_PER_EPOCH) / PAIRS_PER_BATCH)
    total_steps = steps_per_epoch * epochs
    warmup = max(1, int(WARMUP_RATIO * total_steps))
    optimizer = torch.optim.AdamW([{"params": p, "lr": lr} for p, lr in groups], weight_decay=WEIGHT_DECAY)
    lr_factor = lambda s: (s + 1) / warmup if s < warmup else max(0.0, (total_steps - s) / max(1, total_steps - warmup))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    params = [p for p in model.parameters() if p.requires_grad]
    kd = spec.get("kd", {})
    dev_pool = make_small_pool(dev_split, DEV_POOL_CAP)
    print(f"{len(pairs)} training pairs | {steps_per_epoch} steps/epoch | dev pool for selection: {len(dev_pool)} of {len(dev_split.pool_ids)}")

    def sample_neg(qi):
        if hardneg is not None and hardneg[qi]:
            return random.choice(hardneg[qi])
        q = tr.queries[qi]; bad = set(q["rel_idx"]) | {q["self_idx"]}
        while True:
            j = random.randrange(len(tr.pool_ids))
            if j not in bad:
                return j

    history, best, bad_epochs, t0 = [], None, 0, time.time()
    for epoch in range(epochs):
        random.shuffle(pairs)
        use = pairs[:PAIRS_PER_EPOCH]
        model.train()
        if teacher is not None:
            teacher.eval()
        run, cnt = 0.0, 0
        optimizer.zero_grad()
        for b0 in range(0, len(use), PAIRS_PER_BATCH):
            batch = use[b0:b0 + PAIRS_PER_BATCH]
            if len(batch) < 2:
                continue
            qis = [qi for qi, _ in batch]
            doc_idx = [p for _, p in batch] + [sample_neg(qi) for qi in qis]      # positives first, then hard negatives
            q_chunks = [tr.q_chunks[qi] for qi in qis]
            d_chunks = [tr.pool_chunks[d] for d in doc_idx]
            mask = torch.zeros(len(batch), len(doc_idx), dtype=torch.bool)
            for i, qi in enumerate(qis):
                q = tr.queries[qi]; rel = set(q["rel_idx"])
                for j, d in enumerate(doc_idx):
                    if j != i and (d in rel or d == q["self_idx"]):
                        mask[i, j] = True                                           # false negative: do not penalise
            mask = mask.to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                qe = encode_docs_train(model, q_chunks)
                de = encode_docs_train(model, d_chunks)
                if teacher is not None:
                    with torch.no_grad():
                        tq, td = encode_docs_train(teacher, q_chunks), encode_docs_train(teacher, d_chunks)
            logits = ((qe.float() @ de.float().T) / TAU).masked_fill(mask, -1e4)
            labels = torch.arange(len(batch), device=device)
            loss = F.cross_entropy(logits, labels)
            if teacher is not None:
                T = kd.get("T", 2.0); a = kd.get("alpha", 0.5)
                t_logits = ((tq.float() @ td.float().T) / TAU).masked_fill(mask, -1e4)
                kl = F.kl_div(F.log_softmax(logits / T, dim=-1), F.softmax(t_logits / T, dim=-1), reduction="batchmean") * (T * T)
                loss = a * loss + (1 - a) * kl
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(); scheduler.step()
            run += loss.item(); cnt += 1
            if cnt % 50 == 0:
                print(f"  epoch {epoch+1} | step {cnt}/{steps_per_epoch} | running loss {run/cnt:.4f} | {time.time()-t0:.0f}s", flush=True)
        S, _ = dense_scores(model, dev_split, dev_pool)
        res = eval_scores(dev_split, S, dev_pool)
        history.append({"epoch": epoch + 1, "train_loss": run / max(cnt, 1), **res})
        print(f"epoch {epoch+1}/{epochs} | train loss {run/max(cnt,1):.4f} | dev (reduced pool) {fmt(res)} | {time.time()-t0:.0f}s", flush=True)
        if best is None or res["f1"] > best:
            best, bad_epochs = res["f1"], 0
            torch.save(trainable_state(model), paths["model_path"])
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"early stopping at epoch {epoch+1}")
                break
    return history, time.time() - t0

In [ ]:
def build_model_for(spec):
    """Phase 1 backbone + one adaptation technique + the projection head. Returns (model, [(params, lr)])."""
    lm = load_lm(spec["backbone"])
    for p in lm.parameters():
        p.requires_grad = False
    tech, n_prefix = spec["technique"], 0
    if tech == "lorma":
        lm = apply_lorma(lm, spec["targets"], r=spec["r"], alpha=spec["alpha"])
    elif tech == "ia3":
        lm = apply_ia3(lm, N_EMBD)
    elif tech == "partial":
        for blk in lm.h[-spec["n_blocks"]:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in lm.ln_f.parameters():
            p.requires_grad = True
    elif tech == "full":
        for p in lm.parameters():
            p.requires_grad = True
    elif tech == "prefix":
        n_prefix = spec["n_soft"]
        assert n_prefix <= MAX_PREFIX
    else:
        raise ValueError(tech)
    model = DocEncoder(lm, n_prefix=n_prefix).to(device)
    lm_params = [p for p in model.lm.parameters() if p.requires_grad]
    extra = [model.prefix_k, model.prefix_v] if n_prefix else []
    groups = [(list(model.proj.parameters()), spec.get("lr_head", 1e-3))]
    if lm_params or extra:
        groups.append((lm_params + extra, spec["lr"]))
    return model, groups

def build_student(spec):
    model = DocEncoder(build_student_lm(spec["backbone"])).to(device)
    for p in model.parameters():
        p.requires_grad = True
    return model, [(list(model.proj.parameters()), spec.get("lr_head", 1e-3)),
                   (list(model.lm.parameters()), spec["lr"])]

def dev_f1_of(info_path):
    return json.load(open(info_path)).get("dev", {}).get("dense", {}).get("f1", -1)

def pick_best_spec():
    """The best finished experiment among e1..e5 by DENSE dev F1@7 (the primary metric), or TEACHER_EXP."""
    specs = {k: v for k, v in EXPERIMENT_SPECS.items() if k != "e6"}
    if TEACHER_EXP:
        return specs[TEACHER_EXP]
    best, best_spec = -1.0, None
    for k, s in specs.items():
        p = experiment_dirs(s["name"])["run_info_path"]
        if os.path.exists(p) and dev_f1_of(p) > best:
            best, best_spec = dev_f1_of(p), s
    assert best_spec is not None, "no finished experiment found: run e1-e5 first or set TEACHER_EXP / TEACHER_STATE_PATH"
    print("best experiment:", best_spec["name"], f"(dev F1@7 {best:.4f})")
    return best_spec

def load_experiment_model(spec, path=None):
    model, _ = build_model_for(spec)
    load_trainable(model, path or experiment_dirs(spec["name"])["model_path"])
    return model

def run_experiment(spec):
    name = spec["name"]; paths = experiment_dirs(name)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    print(f"===== {name} =====")
    teacher, teacher_name = None, None
    if spec["technique"] == "kd":
        t_spec = pick_best_spec(); teacher_name = t_spec["name"]
        teacher = load_experiment_model(t_spec, TEACHER_STATE_PATH)
        for p in teacher.parameters():
            p.requires_grad = False
        teacher.eval()
        spec = {**spec, "backbone": t_spec["backbone"]}
        model, groups = build_student(spec)
    else:
        model, groups = build_model_for(spec)
    trainable, total = count_params(model)
    proj_params = sum(p.numel() for p in model.proj.parameters())
    print(f"trainable {trainable:,} / {total:,} ({100*trainable/total:.2f}%) | of which projection head {proj_params:,}")

    history, train_secs = train_retriever(model, groups, spec, paths, teacher)
    load_trainable(model, paths["model_path"])
    if teacher is not None:
        del teacher; gc.collect(); torch.cuda.empty_cache()

    # final dev evaluation on the FULL dev pool: dense, TF-IDF, and the fused ranking
    t0 = time.time()
    dev_S, _ = dense_scores(model, dev_split)
    enc_secs = time.time() - t0
    dense_res = eval_scores(dev_split, dev_S)
    alpha, hyb_res = tune_hybrid(dev_split, dev_S, dev_split.tfidf_scores())
    print("FINAL DEV dense :", fmt(dense_res))
    print(f"FINAL DEV hybrid: {fmt(hyb_res)} (alpha {alpha})")

    test_S, _ = dense_scores(model, test_split)
    write_submission(paths["dense_dir"], predictions_from_scores(test_split, test_S), test_split)
    write_submission(paths["hybrid_dir"], predictions_from_scores(test_split, hybrid_scores(test_S, test_split.tfidf_scores(), alpha)), test_split)

    n_docs = len(dev_split.queries) + len(dev_split.pool_ids)
    info = {"experiment": name, "backbone": spec["backbone"], "technique": spec["technique"],
            "settings": {k: v for k, v in spec.items() if k != "name"}, "teacher": teacher_name,
            "trainable_params": trainable, "projection_params": proj_params, "adapter_params": trainable - proj_params,
            "total_params": total, "chunk": [CHUNK_LEN, MAX_CHUNKS], "pairs_per_epoch": PAIRS_PER_EPOCH,
            "hard_negatives": HARD_NEG, "tau": TAU, "epochs_run": len(history), "history": history,
            "dev": {"dense": dense_res, "hybrid": {**hyb_res, "alpha": alpha}},
            "train_seconds": round(train_secs), "dev_encode_seconds": round(enc_secs, 1),
            "docs_per_second": round(n_docs / enc_secs, 1),
            "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
            "checkpoint_mb": round(os.path.getsize(paths["model_path"]) / 1e6, 1)}
    with open(paths["run_info_path"], "w") as f:
        json.dump(info, f, indent=2)
    del model; gc.collect(); torch.cuda.empty_cache()
    return info

print("Runner defined.")

## 13. The six experiments

Hyperparameters are starting points, not tuned values. Test memory with one step if you change `PAIRS_PER_BATCH` (each step encodes 4 x pairs x chunks sequences of `CHUNK_LEN` tokens with gradients).

In [ ]:
EXPERIMENT_SPECS = {
    "e1": dict(name="01_gqa_lorma", backbone="gqa", technique="lorma",
               targets=["q_proj", "k_proj", "v_proj", "c_proj"], r=8, alpha=16, lr=5e-4, epochs=10, patience=5),
    "e2": dict(name="02_gpt2_ia3", backbone="gpt2", technique="ia3", lr=5e-3, epochs=10, patience=5),
    "e3": dict(name="03_swiglu_partial_ft", backbone="swiglu", technique="partial", n_blocks=4, lr=5e-5, epochs=10, patience=5),
    "e4": dict(name="04_prefixlm_prefix_tuning", backbone="prefixlm", technique="prefix", n_soft=32, lr=5e-3, epochs=10, patience=5),
    "e5": dict(name="05_gqa_full_ft", backbone="gqa", technique="full", lr=3e-5, epochs=10, patience=5),
    "e6": dict(name="06_kd_half_layer_student", backbone=None, technique="kd", lr=5e-5, epochs=10, patience=5,
               kd=dict(T=2.0, alpha=0.5)),
}
print({k: v["name"] for k, v in EXPERIMENT_SPECS.items()})

### Experiment 1 -- GQA + LoRMA (compulsory)

In [ ]:
if RUN["e1"]:
    info1 = run_experiment(EXPERIMENT_SPECS["e1"])

### Experiment 2 -- GPT-2 + IA3

In [ ]:
if RUN["e2"]:
    info2 = run_experiment(EXPERIMENT_SPECS["e2"])

### Experiment 3 -- SwiGLU + partial fine-tuning (last 4 blocks)

In [ ]:
if RUN["e3"]:
    info3 = run_experiment(EXPERIMENT_SPECS["e3"])

### Experiment 4 -- Prefix-LM + Prefix Tuning (causal)

In [ ]:
if RUN["e4"]:
    info4 = run_experiment(EXPERIMENT_SPECS["e4"])

### Experiment 5 -- GQA + full fine-tuning

In [ ]:
if RUN["e5"]:
    info5 = run_experiment(EXPERIMENT_SPECS["e5"])

### Experiment 6 -- Knowledge distillation (mandatory)

Teacher = the best finished experiment among 1-5 by **dev F1@7** (or `TEACHER_EXP`), rebuilt and loaded from its saved `best_model.pt`.
Student = the same architecture with 6 of the 12 layers, initialised from every other Phase 1 layer, fully trainable, with its own projection head.
The student learns from the teacher's similarity distribution over the batch's candidates plus the contrastive loss. The student keeps the large embedding matrix,
so its parameter saving is smaller than the layer saving; `run_info.json` has the exact counts, memory and encoding speed.

In [ ]:
if RUN["e6"]:
    info6 = run_experiment(EXPERIMENT_SPECS["e6"])

## 14. Analysis cells (best experiment)

Chunk aggregation (how a long case becomes one representation) and a failure analysis. Both load the best finished experiment's checkpoint.

In [ ]:
best_spec = pick_best_spec()
best_model = load_experiment_model(best_spec)
q_emb, qc = embed_chunks_all(best_model, dev_split.q_chunks)
c_emb, cc = embed_chunks_all(best_model, dev_split.pool_chunks)
print("chunk aggregation on dev (full pool), model:", best_spec["name"])
for mode in ("mean", "first", "maxsim"):
    S = maxsim_scores(q_emb, qc, c_emb, cc) if mode == "maxsim" else aggregate(q_emb, qc, mode) @ aggregate(c_emb, cc, mode).T
    print(f"  {mode:7s}:", fmt(eval_scores(dev_split, S.cpu().numpy())))

In [ ]:
S_dense = (aggregate(q_emb, qc) @ aggregate(c_emb, cc).T).cpu().numpy()
preds = predictions_from_scores(dev_split, S_dense)
gold = dev_split.gold()
hits = {q: len(set(preds[q]) & gold[q]) for q in gold}
print("oracle upper bound:", fmt(oracle_f1_at_k(gold)))
print("queries with 0 hits in the top 7:", sum(h == 0 for h in hits.values()), "of", len(hits))
print("hits per query histogram:", sorted(Counter(hits.values()).items()))
buckets = {"1-3 gold": (1, 3), "4-7 gold": (4, 7), ">7 gold": (8, 10**9)}
for b, (lo, hi) in buckets.items():
    qs = [q for q in gold if lo <= len(gold[q]) <= hi]
    if qs:
        tp = sum(hits[q] for q in qs); ng = sum(len(gold[q]) for q in qs)
        print(f"  {b:9s}: {len(qs):3d} queries | recall@7 {tp/ng:.3f}")
rank_of = []                                   # where do the gold candidates sit in the full ranking?
for qi, q in enumerate(dev_split.queries):
    order = np.argsort(-S_dense[qi]); pos = {int(i): r for r, i in enumerate(order)}
    rank_of += [pos[i] + 1 for i in q["rel_idx"]]
print("median rank of a gold candidate:", int(np.median(rank_of)), "| share of gold within top 7 / 30 / 100:",
      [round(float(np.mean(np.array(rank_of) <= k)), 3) for k in (7, 30, 100)])
del best_model; gc.collect(); torch.cuda.empty_cache()

## 15. Results table (all finished experiments)

In [ ]:
import glob
rows = []
for f in sorted(glob.glob(os.path.join(BASE_OUTPUT_DIR, "*", "run_info.json"))):
    ri = json.load(open(f)); d, h = ri["dev"]["dense"], ri["dev"]["hybrid"]
    rows.append((ri["experiment"], ri["technique"], ri["adapter_params"], ri["trainable_params"], ri["total_params"],
                 d["p"], d["r"], d["f1"], h["f1"], h["alpha"], ri["train_seconds"], ri["docs_per_second"], ri["peak_gpu_gb"], ri["checkpoint_mb"]))
print(f"{'experiment':28s} {'tech':8s} {'adapter':>10s} {'trainable':>11s} {'total':>11s} {'P@7':>6s} {'R@7':>6s} {'F1@7':>6s} {'hybF1':>6s} {'alpha':>5s} {'train_s':>8s} {'docs/s':>7s} {'GB':>5s} {'ckptMB':>7s}")
for r in rows:
    print(f"{r[0]:28s} {r[1]:8s} {r[2]:10,d} {r[3]:11,d} {r[4]:11,d} {r[5]:6.3f} {r[6]:6.3f} {r[7]:6.3f} {r[8]:6.3f} {r[9]:5.1f} {r[10]:8d} {r[11]:7.1f} {r[12]:5.1f} {r[13]:7.1f}")